In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/smoking_prediction.csv")



In [2]:
df.describe()

,ID,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,HDL,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,smoking
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,...,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,26284.994580,44.189300,164.652000,65.878700,3.415468,0.027999,0.027865,0.041026,0.041048,5.063196,...,2.388346,4.791527,0.594483,0.043708,0.018538,1.094032,1.130193,1.668978,0.213120,0.366580
std,16015.070567,12.090086,9.208778,12.831355,0.386760,0.024137,0.024297,0.006325,0.006389,0.570171,...,0.615541,1.709246,0.066414,0.017509,0.017504,0.828010,1.327440,2.110348,0.409516,0.481875
min,0.000000,20.000000,130.000000,30.000000,2.130000,0.000000,0.000000,0.040000,0.040000,2.960000,...,0.170000,0.040000,0.170000,0.040000,0.000000,0.250000,0.040000,0.040000,0.000000,0.000000
25%,12499.750000,40.000000,160.000000,55.000000,3.170000,0.010000,0.010000,0.040000,0.040000,4.670000,...,1.960000,3.830000,0.550000,0.040000,0.010000,0.790000,0.630000,0.710000,0.000000,0.000000
50%,24999.500000,40.000000,165.000000,65.000000,3.420000,0.040000,0.040000,0.040000,0.040000,5.000000,...,2.290000,4.710000,0.590000,0.040000,0.010000,0.960000,0.880000,1.040000,0.000000,0.000000
75%,40034.250000,55.000000,170.000000,75.000000,3.670000,0.040000,0.040000,0.040000,0.040000,5.420000,...,2.750000,5.670000,0.630000,0.040000,0.040000,1.210000,1.290000,1.830000,0.000000,1.000000
max,55690.000000,85.000000,190.000000,135.000000,5.380000,0.380000,0.380000,0.080000,0.080000,10.000000,...,25.750000,77.500000,0.880000,0.250000,0.460000,54.630000,121.420000,41.630000,1.000000,1.000000


### Procesamiento


se borran las columnas smoking, id y oral. Id porque es un identificador unico, oral porque tiene 1 solo valor en todos los registros y smoking porque es la variable a predecir. 

In [3]:
X = df.drop(columns=["ID", "oral", "smoking"])

y = df["smoking"]

print("X:", X.shape)
print("y:", y.shape)

X: (50000, 24)
y: (50000,)


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (35000, 24)
X_test: (15000, 24)
y_train: (35000,)
y_test: (15000,)


In [5]:
print("Entrenamiento:")
print(y_train.value_counts(normalize=True) * 100)

print("\nPrueba:")
print(y_test.value_counts(normalize=True) * 100)

Entrenamiento:
smoking
0    63.342857
1    36.657143
Name: proportion, dtype: float64

Prueba:
smoking
0    63.34
1    36.66
Name: proportion, dtype: float64


#### Codificacion de gender y tartar

In [6]:
X_train.select_dtypes(include="str").columns

Index(['gender', 'tartar'], dtype='str')

In [7]:
from sklearn.preprocessing import OneHotEncoder

columnas_categoricas = ["gender", "tartar"]

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoder.fit(X_train[columnas_categoricas])

print(encoder.categories_)

[array(['F', 'M'], dtype=object), array(['N', 'Y'], dtype=object)]


In [8]:
X_train_encoded = encoder.transform(
    X_train[columnas_categoricas]
)

X_test_encoded = encoder.transform(
    X_test[columnas_categoricas]
)

In [9]:
print(encoder.get_feature_names_out())
print(X_train_encoded[:5])

['gender_F' 'gender_M' 'tartar_N' 'tartar_Y']
[[0. 1. 1. 0.]
 [1. 0. 1. 0.]
 [1. 0. 0. 1.]
 [0. 1. 0. 1.]
 [1. 0. 1. 0.]]


In [10]:
nombres_columnas = encoder.get_feature_names_out()

# Convertir las matrices a DataFrames
df_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=nombres_columnas,
    index=X_train.index
)

df_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=nombres_columnas,
    index=X_test.index
)

In [11]:
X_train_encoded = pd.concat(
    [
        X_train.drop(columns=columnas_categoricas),
        df_train_encoded
    ],
    axis=1
)

X_test_encoded = pd.concat(
    [
        X_test.drop(columns=columnas_categoricas),
        df_test_encoded
    ],
    axis=1
)

Comprobar que no queden variables de texto:

In [12]:
print("Entrenamiento:", X_train_encoded.shape)
print("Prueba:", X_test_encoded.shape)

display(X_train_encoded.head())

Entrenamiento: (35000, 26)
Prueba: (15000, 26)


,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,relaxation,...,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,gender_F,gender_M,tartar_N,tartar_Y
8319,55,165,65,3.54,0.00,0.04,0.04,0.04,5.46,3.71,...,0.04,0.04,1.00,1.13,2.04,0,0.0,1.0,1.0,0.0
7833,60,155,55,3.25,0.01,0.00,0.04,0.04,4.38,2.58,...,0.04,0.01,1.21,1.13,3.17,0,1.0,0.0,1.0,0.0
27390,55,150,50,3.09,0.05,0.04,0.04,0.04,4.92,2.75,...,0.04,0.00,1.50,1.29,1.71,1,1.0,0.0,0.0,1.0
44413,40,165,65,3.13,0.04,0.04,0.04,0.04,5.21,2.83,...,0.04,0.04,0.79,0.79,1.08,1,0.0,1.0,0.0,1.0
24218,40,160,50,2.92,0.04,0.05,0.04,0.04,4.79,2.79,...,0.13,0.00,0.67,0.38,0.33,0,1.0,0.0,1.0,0.0


### Escalado

In [13]:
columnas_numericas = X_train.drop(
    columns=columnas_categoricas
).columns.tolist()

print(columnas_numericas)

['age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)', 'eyesight(right)', 'hearing(left)', 'hearing(right)', 'systolic', 'relaxation', 'fasting blood sugar', 'Cholesterol', 'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'Urine protein', 'serum creatinine', 'AST', 'ALT', 'Gtp', 'dental caries']


In [14]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

scaler.fit(X_train_encoded[columnas_numericas])

,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array).",True
,"clip clip: bool, default=FalseSet to True to clip transformed values of held-out data toprovided `feature_range`.Since this parameter will clip values, `inverse_transform` may notbe able to restore the original data... note:: Setting `clip=True` does not prevent feature drift (a distribution shift between training and test data). The transformed values are clipped to the `feature_range`, which helps avoid unintended behavior in models sensitive to out-of-range inputs (e.g. linear models). Use with care, as clipping can distort the distribution of test data... versionadded:: 0.24",False
Name,Type,Value
"data_max_ data_max_: ndarray of shape (n_features,)Per feature maximum seen in the data.. versionadded:: 0.17 *data_max_*","ndarray[float64](22,)","[ 85. ,190. ,135. ,...,121.42, 41.63, 1. ]"
"data_min_ data_min_: ndarray of shape (n_features,)Per feature minimum seen in the data.. versionadded:: 0.17 *data_min_*","ndarray[float64](22,)","[ 20. ,130. , 30. ,..., 0.08, 0.08, 0. ]"
"data_range_ data_range_: ndarray of shape (n_features,)Per feature range ``(data_max_ - data_min_)`` seen in the data.. versionadded:: 0.17 *data_range_*","ndarray[float64](22,)","[ 65. , 60. ,105. ,...,121.34, 41.55, 1. ]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](22,)","['age','height(cm)','weight(kg)',...,'ALT','Gtp','dental caries']"
"min_ min_: ndarray of shape (n_features,)Per feature adjustment for minimum. Equivalent to``min - X.min(axis=0) * self.scale_``","ndarray[float64](22,)","[-0.31,-2.17,-0.29,...,-0. ,-0. , 0. ]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,22
"n_samples_seen_ n_samples_seen_: intThe number of samples processed by the estimator.It will be reset on new calls to fit, but increments across``partial_fit`` calls.",int,35000


In [15]:
X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()

X_train_scaled[columnas_numericas] = scaler.transform(
    X_train_encoded[columnas_numericas]
)

X_test_scaled[columnas_numericas] = scaler.transform(
    X_test_encoded[columnas_numericas]
)

In [16]:
print("Entrenamiento:", X_train_scaled.shape)
print("Prueba:", X_test_scaled.shape)

display(X_train_scaled.head())

Entrenamiento: (35000, 26)
Prueba: (15000, 26)


,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,relaxation,...,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,gender_F,gender_M,tartar_N,tartar_Y
8319,0.538462,0.583333,0.333333,0.433846,0.000000,0.105263,0.0,0.0,0.402576,0.462585,...,0.000000,0.095238,0.016604,0.008653,0.047172,0.0,0.0,1.0,1.0,0.0
7833,0.615385,0.416667,0.238095,0.344615,0.026316,0.000000,0.0,0.0,0.228663,0.206349,...,0.000000,0.023810,0.021253,0.008653,0.074368,0.0,1.0,0.0,1.0,0.0
27390,0.538462,0.333333,0.190476,0.295385,0.131579,0.105263,0.0,0.0,0.315620,0.244898,...,0.000000,0.000000,0.027673,0.009972,0.039230,1.0,1.0,0.0,0.0,1.0
44413,0.307692,0.583333,0.333333,0.307692,0.105263,0.105263,0.0,0.0,0.362319,0.263039,...,0.000000,0.095238,0.011955,0.005851,0.024067,1.0,0.0,1.0,0.0,1.0
24218,0.307692,0.500000,0.190476,0.243077,0.105263,0.131579,0.0,0.0,0.294686,0.253968,...,0.428571,0.000000,0.009298,0.002472,0.006017,0.0,1.0,0.0,1.0,0.0


Revisamos los mimnimos y maximos para cada variable

In [17]:
print("Mínimos de entrenamiento:")
print(X_train_scaled[columnas_numericas].min())

print("\nMáximos de entrenamiento:")
print(X_train_scaled[columnas_numericas].max())

Mínimos de entrenamiento:
age                    0.0
height(cm)             0.0
weight(kg)             0.0
waist(cm)              0.0
eyesight(left)         0.0
eyesight(right)        0.0
hearing(left)          0.0
hearing(right)         0.0
systolic               0.0
relaxation             0.0
fasting blood sugar    0.0
Cholesterol            0.0
triglyceride           0.0
HDL                    0.0
LDL                    0.0
hemoglobin             0.0
Urine protein          0.0
serum creatinine       0.0
AST                    0.0
ALT                    0.0
Gtp                    0.0
dental caries          0.0
dtype: float64

Máximos de entrenamiento:
age                    1.0
height(cm)             1.0
weight(kg)             1.0
waist(cm)              1.0
eyesight(left)         1.0
eyesight(right)        1.0
hearing(left)          1.0
hearing(right)         1.0
systolic               1.0
relaxation             1.0
fasting blood sugar    1.0
Cholesterol            1.0
triglyceride  

In [18]:
print("Nulos entrenamiento:", X_train_scaled.isnull().sum().sum())
print("Nulos prueba:", X_test_scaled.isnull().sum().sum())

print("Mismas columnas:",
      X_train_scaled.columns.equals(X_test_scaled.columns))

Nulos entrenamiento: 0
Nulos prueba: 0
Mismas columnas: True


### Guardado de datos


In [19]:
import joblib
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)
Path("../models").mkdir(parents=True, exist_ok=True)

# Conjuntos de entrenamiento y prueba
joblib.dump(X_train_scaled, "../data/processed/X_train.joblib")
joblib.dump(X_test_scaled, "../data/processed/X_test.joblib")
joblib.dump(y_train, "../data/processed/y_train.joblib")
joblib.dump(y_test, "../data/processed/y_test.joblib")

# Transformadores
joblib.dump(encoder, "../models/encoder.joblib")
joblib.dump(scaler, "../models/scaler.joblib")

# Columnas utilizadas
joblib.dump(columnas_categoricas, "../models/columnas_categoricas.joblib")
joblib.dump(columnas_numericas, "../models/columnas_numericas.joblib")

['../models/columnas_numericas.joblib']